<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z327_GARCH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GARCH — Generalized AutoRegressive Conditional Heteroskedasticity

## ¿Por qué GARCH para ventas?

HAR y ARIMA modelan la **media condicional** de la serie.
GARCH agrega una capa: modela también la **varianza condicional** (la volatilidad cambia en el tiempo).

Para ventas esto captura:
- Períodos de alta incertidumbre (lanzamientos, quiebres de stock) → varianza alta
- Períodos estables → varianza baja
- La predicción puntual viene de la media condicional, pero la varianza condicional puede usarse para ponderar

## Estructura del modelo: ARMA(p,q) + GARCH(1,1)

```
Media:     tn_t = c + φ1·tn_{t-1} + ... + θ1·ε_{t-1} + ... + ε_t
Varianza:  σ²_t = ω + α1·ε²_{t-1} + β1·σ²_{t-1}
```

- **α1** (ARCH term): cuánto impacta un shock reciente en la volatilidad futura
- **β1** (GARCH term): cuánto persiste la volatilidad pasada
- α1 + β1 < 1 → proceso estacionario en varianza

## Variantes que probamos

| Variante | Media | Varianza | Intuición |
|---|---|---|---|
| GARCH(1,1) | AR(1) | GARCH(1,1) | Baseline clásico |
| GJR-GARCH | AR(1) | GJR(1,1) | Asimetría: shocks negativos tienen más impacto |
| EGARCH | AR(1) | EGARCH(1,1) | Varianza en log → siempre positiva, asimétrico |

## Pipeline

1. Test de Racha → series con estructura van a GARCH, el resto a promedio 12m
2. Ajustar ARMA(1,0)+GARCH(1,1) por producto
3. Predicción recursiva: forecast t+1, append, forecast t+2
4. Submit a Kaggle

## 0.1 Init ambiente Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Setup

In [ ]:
!pip install uv
!uv pip install -q kaggle arch

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
  import os
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  os.system(comando)

In [ ]:
import os
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from scipy.stats import runstest_1samp
from arch import arch_model

import warnings
warnings.filterwarnings('ignore')

Por favor, cargar aqui SU semilla primigenia.

In [ ]:
PARAM = {
  'experimento':        'GARCH-01',
  'kaggle_competition': 'labo-iii-2026-rosario',
  'semilla_primigenia': 102191,
  'alpha_runs':         0.05,
  # variante GARCH: 'garch', 'gjr', 'egarch'
  'variante':           'garch',
  # orden media: AR(p) + MA(q)
  'ar_p': 1,
  'ma_q': 0,
  # orden varianza: GARCH(p=1, q=1)
  'garch_p': 1,
  'garch_q': 1,
  # distribucion de errores: 'normal', 't', 'skewt'
  'dist': 'normal',
}

In [ ]:
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

# 2  Preparacion de datos

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")

tb_ventas = tb_ventas.join(tb_apredecir, on="product_id", how="inner").sort(["product_id", "periodo"])
print(f"{tb_ventas.height} filas, {tb_ventas['product_id'].n_unique()} productos")

# 3  Test de Racha por producto

Filtramos las series sin estructura temporal — para esas usamos directamente el promedio 12m.
Las series con estructura (p < alpha) van al modelo GARCH.

In [ ]:
productos = tb_apredecir["product_id"].to_list()
runs_resultados = []

for producto in productos:
    serie = (
        tb_ventas.filter(pl.col("product_id") == producto)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )
    try:
        _, pvalue = runstest_1samp(serie, cutoff='median')
        tiene_estructura = bool(pvalue < PARAM['alpha_runs'])
    except Exception:
        pvalue = np.nan
        tiene_estructura = False

    promedio_12m = float(serie[-12:].mean()) if len(serie) >= 12 else float(serie.mean())

    runs_resultados.append({
        'product_id':       producto,
        'tiene_estructura': tiene_estructura,
        'promedio_12m':     promedio_12m
    })

tb_runs = pl.DataFrame(runs_resultados)

n_estructura = tb_runs["tiene_estructura"].sum()
print(f"Con estructura (→ GARCH) : {n_estructura}")
print(f"Sin estructura (→ prom12m): {len(productos) - n_estructura}")

# 4  Ajuste GARCH por producto

Usamos la librería `arch` de Kevin Sheppard (estándar en Python para GARCH).

### Variantes disponibles via `PARAM['variante']`

| Variante | Ecuación varianza | Cuándo usarla |
|---|---|---|
| `'garch'` | σ²_t = ω + α·ε²_{t-1} + β·σ²_{t-1} | Baseline simétrico |
| `'gjr'` | Agrega γ·ε²_{t-1}·I(ε_{t-1}<0) | Shocks negativos → más volatilidad |
| `'egarch'` | log(σ²_t) = ω + α·\|z_{t-1}\| + γ·z_{t-1} + β·log(σ²_{t-1}) | Asimetría + siempre positivo |

### Predicción recursiva

El forecast de `arch` da la media condicional E[tn_{t+h}] para h pasos.
Pedimos `horizon=2` directamente — no hay que hacer el paso recursivo a mano.

In [ ]:
estructura_dict = dict(zip(
    tb_runs["product_id"].to_list(),
    tb_runs["tiene_estructura"].to_list()
))
promedio_dict = dict(zip(
    tb_runs["product_id"].to_list(),
    tb_runs["promedio_12m"].to_list()
))

# mapeo de variante a parametros arch
VARIANTE_MAP = {
    'garch':  {'vol': 'GARCH', 'p': PARAM['garch_p'], 'q': PARAM['garch_q']},
    'gjr':    {'vol': 'GARCH', 'p': PARAM['garch_p'], 'o': 1, 'q': PARAM['garch_q']},
    'egarch': {'vol': 'EGARCH','p': PARAM['garch_p'], 'q': PARAM['garch_q']},
}
vol_params = VARIANTE_MAP[PARAM['variante']]

resultados = []
modelos_ajustados = {}  # para inspeccion posterior
n_ok = 0
n_fallback = 0

for producto in productos:
    serie = (
        tb_ventas.filter(pl.col("product_id") == producto)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )

    fallback = promedio_dict[producto]

    if not estructura_dict.get(producto, False):
        pred_2 = fallback
        metodo = 'promedio12m'
        n_fallback += 1
    else:
        try:
            # rescalamos para mejorar convergencia numerica
            escala = serie.mean() if serie.mean() > 0 else 1.0
            serie_sc = serie / escala

            am = arch_model(
                serie_sc,
                mean='ARX',
                lags=PARAM['ar_p'],
                **vol_params,
                dist=PARAM['dist']
            )
            res = am.fit(disp='off', show_warning=False)
            modelos_ajustados[producto] = res

            # forecast a horizon=2 — el paso 2 es nuestro 202002
            fc = res.forecast(horizon=2, reindex=False)
            pred_2_sc = float(fc.mean.values[0, 1])  # h=2
            pred_2 = max(pred_2_sc * escala, 0.0)
            metodo = f'GARCH_{PARAM["variante"]}'
            n_ok += 1

        except Exception as e:
            pred_2 = fallback
            metodo = 'promedio12m_error'
            n_fallback += 1

    resultados.append({'product_id': producto, 'tn': pred_2, 'metodo': metodo})

print(f"GARCH ok: {n_ok}  |  fallback: {n_fallback}")

# 5  Resumen de resultados

In [ ]:
tb_resultado = pl.DataFrame(resultados)
display(tb_resultado.group_by("metodo").agg(pl.len().alias("n")).sort("n", descending=True))

## 5.1  Inspeccion de parámetros GARCH

Miramos α1 + β1 de cada modelo ajustado:
- Cerca de 1 → alta persistencia de volatilidad (común en series financieras)
- Cerca de 0.5 → volatilidad se disipa rápido

Para ventas de productos es esperable que α1 + β1 sea bajo o que el GARCH sea casi ARCH puro.

In [ ]:
params_list = []

for pid, res in modelos_ajustados.items():
    p = res.params
    alpha1 = p.get('alpha[1]', np.nan)
    beta1  = p.get('beta[1]',  np.nan)
    params_list.append({
        'product_id': pid,
        'alpha1': alpha1,
        'beta1':  beta1,
        'persistencia': alpha1 + beta1
    })

if params_list:
    tb_params = pl.DataFrame(params_list)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(tb_params['alpha1'].drop_nulls().to_numpy(), bins=30, color='steelblue', edgecolor='white')
    axes[0].set_title('Distribución α1 (ARCH term)')
    axes[0].set_xlabel('α1')

    axes[1].hist(tb_params['persistencia'].drop_nulls().to_numpy(), bins=30, color='orange', edgecolor='white')
    axes[1].axvline(1.0, color='red', linestyle='--', label='persistencia=1')
    axes[1].set_title('Distribución α1+β1 (persistencia)')
    axes[1].set_xlabel('α1 + β1')
    axes[1].legend()

    plt.suptitle(f'Parámetros GARCH — variante: {PARAM["variante"]}', fontsize=11)
    plt.tight_layout()
    plt.show()

    print(f"Persistencia media: {tb_params['persistencia'].mean():.3f}")
    print(f"% con persistencia > 0.9: {(tb_params['persistencia'] > 0.9).mean() * 100:.1f}%")

## 5.2  Fitted vs real — primeros 6 productos con GARCH

In [ ]:
pids_plot = list(modelos_ajustados.keys())[:6]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, pid in enumerate(pids_plot):
    serie = (
        tb_ventas.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )
    res = modelos_ajustados[pid]

    # media condicional fitted (en-sample)
    fitted_mean = res.conditional_volatility  # esto es sigma; para la media:
    # usamos resid + fitted del modelo de media
    fitted_mu = serie - res.resid  # aproximacion: serie - residuo

    axes[i].plot(range(len(serie)), serie, 'o-', color='steelblue',
                 label='real', markersize=3, linewidth=1.2)
    axes[i].plot(range(len(fitted_mu)), fitted_mu, '--', color='orange',
                 label='fitted media', linewidth=1.2)

    # banda ±1 sigma
    sigma = res.conditional_volatility
    axes[i].fill_between(
        range(len(sigma)),
        fitted_mu - sigma,
        fitted_mu + sigma,
        alpha=0.2, color='orange', label='±1σ'
    )
    axes[i].set_title(f"product_id {pid}", fontsize=9)
    axes[i].legend(fontsize=7)

fig.suptitle(f'GARCH ({PARAM["variante"]}): media condicional ± volatilidad', fontsize=11)
plt.tight_layout()
plt.show()

# 6  Armado del submit

In [ ]:
tb_final = tb_resultado.select(["product_id", "tn"])

# negativos a cero
tb_final = tb_final.with_columns(
    pl.when(pl.col('tn') < 0).then(0.0).otherwise(pl.col('tn')).alias('tn')
)

display(tb_final)
print(f"Total: {tb_final.height}  |  Nulls: {tb_final['tn'].is_null().sum()}")

# 7  Submit a Kaggle

In [ ]:
archivo = f"GARCH_{PARAM['variante']}.csv"
mensaje = f"ARMA({PARAM['ar_p']},{PARAM['ma_q']})+{PARAM['variante'].upper()}({PARAM['garch_p']},{PARAM['garch_q']}) dist={PARAM['dist']} + Racha"

tb_final.write_csv(archivo)
kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje)

# 8  Qué probar si el score no mejora

| Cambio | Donde | Por qué |
|---|---|---|
| `'variante': 'gjr'` | PARAM | Asimetría: caídas de ventas generan más incertidumbre que subidas |
| `'variante': 'egarch'` | PARAM | log-varianza → siempre positivo, más estable numéricamente |
| `'dist': 't'` | PARAM | Distribución t-Student → colas pesadas, mejor si hay outliers |
| `'dist': 'skewt'` | PARAM | t asimétrica → distribución sesgada de ventas |
| `'ar_p': 2` | PARAM | Más memoria en la media condicional |
| `'ma_q': 1` | PARAM | ARMA(1,1) — captura ruido de corto plazo |
| `'alpha_runs': 0.10` | PARAM | Más series van a GARCH (menos fallback) |
| Transformar con `log1p` antes de ajustar | sección 4 | Estabiliza series con outliers, GARCH en log-ventas |
| Usar varianza condicional como peso en ensemble | post-submit | Combinar GARCH (media) con HAR usando 1/σ² como peso |